# ED.03.02 - RDD

Gustavo Adalid Catalan Carmen - 125052712 - IDIA224

In [1]:
from pyspark.sql import SparkSession
from pyspark import StorageLevel
import shutil

spark = (SparkSession.builder
         .master("local[*]")
         .appName("RDD")
         .config("spark.ui.showConsoleProgress", "false")
         .getOrCreate())
sc = spark.sparkContext
sc.setLogLevel("ERROR")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/22 23:45:34 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


## Ejercicio 1: Word Count básico

In [2]:
texto = ["hola mundo", "hola spark", "mundo spark"]
conteo = (sc.parallelize(texto)
          .flatMap(lambda l: l.split(" "))
          .map(lambda p: (p, 1))
          .reduceByKey(lambda a, b: a + b))
print(conteo.collect())

[('spark', 2), ('mundo', 2), ('hola', 2)]


## Ejercicio 2: Análisis de ventas

In [3]:
ventas = sc.parallelize([("Madrid", 100), ("Barcelona", 200), ("Madrid", 50)])
totales = ventas.reduceByKey(lambda a, b: a + b)
print("Totales:", totales.collect())

mayores = totales.filter(lambda x: x[1] > 100)
print("Mayores a 100:", mayores.collect())

ordenado = mayores.sortBy(lambda x: x[1], ascending=False)
print("Ordenado:", ordenado.collect())

Totales: [('Barcelona', 200), ('Madrid', 150)]
Mayores a 100: [('Barcelona', 200), ('Madrid', 150)]
Ordenado: [('Barcelona', 200), ('Madrid', 150)]


## Ejercicio 3: Filtrado y transformación

In [4]:
precios_usd = sc.parallelize([10, 25, 40, 5, 100, 15])
precios_eur = (precios_usd
               .filter(lambda x: x >= 20)
               .map(lambda x: x * 0.9))
print(precios_eur.collect())

[22.5, 36.0, 90.0]


## Ejercicio 4: Procesamiento de CSV

In [5]:
lineas = sc.parallelize(["Leche,Huevos,Pan", "Manzanas,Peras", "Leche,Chocolate"])
productos = lineas.flatMap(lambda l: l.split(","))
unicos = productos.distinct()
print("Productos:", productos.collect())
print("Únicos:", unicos.collect())
print("Cantidad de únicos:", unicos.count())

Productos: ['Leche', 'Huevos', 'Pan', 'Manzanas', 'Peras', 'Leche', 'Chocolate']
Únicos: ['Pan', 'Peras', 'Chocolate', 'Manzanas', 'Leche', 'Huevos']
Cantidad de únicos: 6


## Ejercicio 5: Análisis de logs

In [6]:
logs = sc.textFile("logs.txt")
por_ip = logs.map(lambda l: (l.split(",")[0], 1)).reduceByKey(lambda a, b: a + b)
print("Peticiones por IP:", por_ip.collect())

repetidas = por_ip.filter(lambda x: x[1] > 1)
print("IPs con más de 1 petición:", repetidas.collect())

shutil.rmtree("salida_ips", ignore_errors=True)
repetidas.coalesce(1).saveAsTextFile("salida_ips")
print(open("salida_ips/part-00000").read())

Peticiones por IP: [('192.168.1.1', 2), ('192.168.1.2', 1)]
IPs con más de 1 petición: [('192.168.1.1', 2)]
('192.168.1.1', 2)



## Ejercicio 6: Pipeline y Shuffle

In [7]:
ventas = sc.parallelize([("Madrid", 100), ("Barcelona", 200), ("Madrid", 50), ("Valencia", 50)])
resultado = (ventas
             .filter(lambda x: x[1] >= 60)
             .reduceByKey(lambda a, b: a + b)
             .sortBy(lambda x: x[1], ascending=False))
print(resultado.collect())

[('Barcelona', 200), ('Madrid', 100)]


## Ejercicio 7: Diferencia map vs flatMap

In [8]:
frases = sc.parallelize(["Hola mundo", "Apache Spark es guay"])
print("map:", frases.map(lambda f: f.split(" ")).collect())
print("flatMap:", frases.flatMap(lambda f: f.split(" ")).collect())

map: [['Hola', 'mundo'], ['Apache', 'Spark', 'es', 'guay']]
flatMap: ['Hola', 'mundo', 'Apache', 'Spark', 'es', 'guay']


`map` devuelve una lista por cada frase (listas anidadas). `flatMap` aplana el resultado y devuelve una sola lista de palabras.

## Ejercicio 8: Optimización

In [9]:
datos = sc.parallelize(range(1000000)).map(lambda x: x * 2).filter(lambda x: x % 3 == 0)
datos.persist(StorageLevel.MEMORY_AND_DISK)

print("count:", datos.count())
print("suma:", datos.sum())
print("máximo:", datos.max())
print("mínimo:", datos.min())
print("primeros 3:", datos.take(3))

datos.unpersist()
print("RDD liberado")

count: 333334
suma: 333333666666
máximo: 1999998
mínimo: 0
primeros 3: [0, 6, 12]
RDD liberado


Si un RDD se usa varias veces hay que guardarlo con `cache()` o `persist(StorageLevel.MEMORY_AND_DISK)`. Así Spark no recalcula todo el linaje en cada acción. Al terminar se libera con `unpersist()`.

## Proyecto práctico: Análisis de calificaciones

In [10]:
def parsear(linea):
    try:
        nombre, nota = linea.split(",")
        return [(nombre.strip(), float(nota))]
    except ValueError:
        return []

mates = sc.textFile("notas_mates.txt").flatMap(parsear)
fisica = sc.textFile("notas_fisica.txt").flatMap(parsear)
ingles = sc.textFile("notas_ingles.txt").flatMap(parsear)

todas = mates.union(fisica).union(ingles)
print("Total de notas:", todas.count())

Total de notas: 7


### Promedio, máximo y mínimo por alumno

In [11]:
promedios = (todas
             .aggregateByKey((0.0, 0),
                             lambda acc, v: (acc[0] + v, acc[1] + 1),
                             lambda a, b: (a[0] + b[0], a[1] + b[1]))
             .mapValues(lambda x: round(x[0] / x[1], 2)))

maximos = todas.reduceByKey(max)
minimos = todas.reduceByKey(min)

print("Promedios:", promedios.sortBy(lambda x: x[1], ascending=False).collect())
print("Máximos:", maximos.sortByKey().collect())
print("Mínimos:", minimos.sortByKey().collect())

Promedios: [('Angel', 6.33), ('Ramon', 4.5), ('Maria', 3.67)]
Máximos: [('Angel', 9.0), ('Maria', 6.0), ('Ramon', 4.5)]
Mínimos: [('Angel', 4.0), ('Maria', 2.0), ('Ramon', 4.5)]


### Estadísticas por asignatura

In [12]:
for nombre, rdd in [("Matemáticas", mates), ("Física", fisica), ("Inglés", ingles)]:
    notas = rdd.values()
    print(nombre, "- promedio:", round(notas.mean(), 2),
          "máximo:", notas.max(), "mínimo:", notas.min())

Matemáticas - promedio: 4.17 máximo: 6.0 mínimo: 2.0
Física - promedio: 6.0 máximo: 9.0 mínimo: 3.0
Inglés - promedio: 5.0 máximo: 6.0 mínimo: 4.0


### Operaciones de conjuntos

In [13]:
en_mates = mates.keys()
en_fisica = fisica.keys()
en_ingles = ingles.keys()

en_todas = en_mates.intersection(en_fisica).intersection(en_ingles)
en_alguna = en_mates.union(en_fisica).union(en_ingles).distinct()
incompletos = en_alguna.subtract(en_todas)

print("Alumnos en las tres asignaturas:", sorted(en_todas.collect()))
print("Alumnos en al menos una:", sorted(en_alguna.collect()))
print("Alumnos que faltan en alguna:", sorted(incompletos.collect()))

Alumnos en las tres asignaturas: ['Angel', 'Maria']
Alumnos en al menos una: ['Angel', 'Maria', 'Ramon']
Alumnos que faltan en alguna: ['Ramon']


### Aprobados (promedio >= 5)

In [14]:
aprobados = promedios.filter(lambda x: x[1] >= 5)
reprobados = promedios.filter(lambda x: x[1] < 5)
print("Aprobados:", aprobados.collect())
print("Reprobados:", reprobados.collect())

Aprobados: [('Angel', 6.33)]
Reprobados: [('Ramon', 4.5), ('Maria', 3.67)]


## Pregunta de reflexión: ¿cuándo usaría un RDD en lugar de un DataFrame?

Usaría un RDD cuando los datos no tienen una estructura fija, por ejemplo logs de un servidor con líneas de formatos distintos. Ahí puedo leer el texto con `textFile` y usar `flatMap` para parsear cada línea y descartar las malformadas, algo que no se acomoda bien a las columnas de un DataFrame. En cambio, para datos estructurados como CSV, JSON o Parquet usaría DataFrames, porque el optimizador Catalyst los ejecuta más rápido.